In [0]:
from pyspark.sql.functions import regexp_replace, col,trim

In [0]:
-# [ ]  Analyze data quality using SQL and List all identified issues
-# [ ]  Find duplicates
-# [ ]  Validate string values: Check extra spaces, Identify abbreviations to normalize
-# [ ]  Validate dates values: Check Data Type, check the format, handle missing values
-# [ ]  Validate numeric values
-# [ ]  Standardize business key IDs to ensure tables can be joined correctly.
-# [ ]  Check the name of columns and table and make a plan how to rename them to something friendly.

###Analyze data quality using SQL and List all identified issues

In [0]:
%sql
Select * FROM workspace.bronze_schema.cust_info_delta

### Read data Bronze Table and Load it into a DataFrame

In [0]:
df = spark.table('workspace.bronze_schema.cust_info_delta')
df.display()


### TRANSFORMATIONS


### 1.0 RENAME COLUMNS

In [0]:
df = df.withColumnRenamed('cst_id', 'customer_id')\
    .withColumnRenamed('cst_key', 'customer_key')\
    .withColumnRenamed('cst_firstname', 'customer_first_name')\
    .withColumnRenamed('cst_lastname', 'customer_last_name')\
    .withColumnRenamed('cst_marital_status', 'customer_marital_status')\
    .withColumnRenamed('cst_gndr', 'customer_gender')\
    .withColumnRenamed('cst_create_date', 'customer_create_date')

df.display()

### RENAME VALUES OF SOME COLUMNS(MARITAL STATUS, AND GENDER)

In [0]:


df = df.withColumn('customer_marital_status', regexp_replace(col('customer_marital_status'), 'S', 'small'))\
    .withColumn('customer_marital_status', regexp_replace(col('customer_marital_status'), 'M', 'medium'))\
    .withColumn('customer_marital_status', regexp_replace(col('customer_marital_status'), 'L', 'large'))\
    .withColumn('customer_marital_status', regexp_replace(col('customer_marital_status'), 'XL', 'extra large'))\
    .withColumn('customer_gender', regexp_replace(col('customer_gender'), 'M', 'Male'))\
    .withColumn('customer_gender', regexp_replace(col('customer_gender'), 'F', 'Female'))
    




### TRIM FOR SPACES

In [0]:
df = df.withColumn('customer_first_name',trim(col('customer_first_name')))\
    .withColumn('customer_last_name',trim(col('customer_last_name')))\
    .withColumn('customer_marital_status',trim(col('customer_marital_status')))

df.display()


    

### FILL NOT_AVAILABLE IN NULL VALUES

In [0]:
df = df.fillna('Not_Available', subset=['customer_gender'])
df.display()

### DROP ANY ROW THAT DOESN'T HAVE ANY VALUE

In [0]:
# Using dropna('all', thresh=1) will drop all rows with any null values
df.dropna('all').display()

In [0]:
### WRITE TO SILVER TABLE
df.write.mode('overwrite').saveAsTable('silver_schema.customer_info_silver')